# Taller de la semana 5 — SOLUCIÓN DE REFERENCIA (solo para el docente)

**Enunciado:** [`README.md`](README.md) · **Plan de clase:** [`../../weeks/week-05.md`](../../weeks/week-05.md) · **Motor proporcionado:** [`connect4.py`](connect4.py)

Este notebook presenta la **explicación didáctica** del taller del torneo de la semana 5.
**Importa el código de referencia de [`solution.py`](solution.py)** sin volver a copiarlo,
por lo que el agente mostrado es exactamente el que evalúan las pruebas proporcionadas.
La garantía de corrección se comprueba por separado: el comando documentado
`python3 studios/_verify_solutions.py week-05` ejecuta `test_alphabeta.py`
sin modificaciones contra `solution.py`.

> No distribuir. Se excluye de la versión para estudiantes mediante `studios/.gitignore`.


In [1]:
# --- Preparación: añade la carpeta de la semana (solution/connect4) y la raíz del repositorio
# (aicourse) a la ruta de importación desde las ubicaciones previstas. ---
import sys, pathlib
# POO: pathlib.Path es una clase; cwd() devuelve un objeto con la ruta del directorio actual.
# / combina rutas y exists() comprueba la existencia de un archivo mediante el objeto Path.
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-05"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        # Método de list: añade estas carpetas al principio de la búsqueda de módulos de Python.
        sys.path.insert(0, p)

import inspect, time
from math import inf
import connect4
from connect4 import Connect4, minimax, alphabeta, play, MAX, MIN, CENTER_FIRST
import solution

## `order_moves` — Primero las mejores jugadas para aprovechar la poda

El mejor caso O(b^(m/2)) de α-β, que duplica la profundidad alcanzable, requiere examinar
primero jugadas fuertes. La referencia usa un orden estático que comienza por el centro y lo
filtra para conservar solo las columnas legales: una *permutación* de `board.legal_moves()`.


In [2]:
# getsource obtiene el código de la función importada, incluidos sus comentarios, sin copiarlo aquí.
print(inspect.getsource(solution.order_moves))
# Instanciación y métodos: b es un objeto tablero y push coloca fichas modificando ese mismo objeto.
b = Connect4(); b.push(3, MAX); b.push(3, MIN)
# order_moves es una función del módulo solution, no un método de b; recibe el objeto como argumento.
order = solution.order_moves(b)
print("order :", order)
print("legal :", b.legal_moves())
# Aserción equivalente a la prueba: exactamente las columnas legales, con otro orden.
assert sorted(order) == sorted(b.legal_moves())
print("ok  order_moves returns the legal columns, reordered (center-first)")

def order_moves(board):
    """Return this board's legal columns in the order α-β should try them.

    Center-first (``CENTER_FIRST``) is a strong *static* order for Connect-4 —
    central columns take part in more winning lines — so we simply filter the
    static priority list down to the columns that are actually playable. This
    is a legal permutation of ``board.legal_moves()``.
    """
    return [c for c in CENTER_FIRST if c in board.legal_moves()]

order : [3, 2, 4, 1, 5, 0, 6]
legal : [0, 1, 2, 3, 4, 5, 6]
ok  order_moves returns the legal columns, reordered (center-first)


## `choose_move` — Profundización iterativa para mantener una respuesta disponible

Busca a profundidad 1, después 2, luego 3, etc.; conserva la mejor jugada de la última
profundidad **completada** y detente al agotar el tiempo. El torneo depende de tres condiciones:
devolver una columna legal, respetar el presupuesto temporal y conservar la jugada de la
última profundidad completa. La referencia también aplica un orden dinámico sencillo:
pone primero la mejor jugada de la iteración anterior.


In [3]:
print(inspect.getsource(solution.choose_move))

# Reproduce test_choose_move_returns_legal_column.
# La segunda expresión usa lambdas anidadas: crea un tablero, coloca dos fichas centrales
# y devuelve el objeto b como último elemento de una tupla, seleccionado con [-1].
for bd in (Connect4(),
           (lambda: (lambda b: (b.push(3,MAX), b.push(3,MIN), b)[-1])(Connect4()))()):
    mv = solution.choose_move(bd, 300)
    assert mv in bd.legal_moves()
print("ok  choose_move always returns a legal column")

# Reproduce test_choose_move_respects_the_time_budget.
b = Connect4(); b.push(3, MAX); b.push(3, MIN)
# perf_counter devuelve segundos; después se multiplican por 1000 para obtener milisegundos.
# La referencia solo comprueba su reloj entre profundidades y puede superar 300 ms en una iteración.
t0 = time.perf_counter(); mv = solution.choose_move(b, 300)
elapsed = (time.perf_counter() - t0) * 1000
print(f"chose col {mv} in {elapsed:.0f} ms on a 300 ms budget")
# Esta prueba permite 2000 ms adicionales: no demuestra el límite estricto de 300 ms del torneo.
assert elapsed < 300 + 2000, "budget blown -> forfeits in the tournament"
print("ok  respected the budget")

def choose_move(board, time_budget_ms) -> int:
    """Return the column MAX should play, using iterative deepening + α-β.

    Contract (the tournament relies on all three):
      1. ALWAYS return a legal column, even if the budget is tiny — seed `best`
         with a legal move before the loop so you never return None.
      2. Never exceed `time_budget_ms` by much — check the clock between depths
         and stop; do not start a depth you cannot afford.
      3. Keep the best move from the last COMPLETED depth.

    We also apply the cheap dynamic-ordering trick the README recommends: after
    each completed depth, move the best column to the front of the order so the
    next (deeper) iteration examines it first and prunes harder.
    """
    deadline = time.perf_counter() + time_budget_ms / 1000.0
    order = order_moves(board)
    best = order[0]                       # rule 1: always have a legal move
    for depth in range(1, 43):
        if time.perf_counter() >= deadline:
 

### El agente reconoce tácticas inmediatas

Debe aprovechar una victoria inmediata y bloquear una amenaza inmediata.
Una victoria propia se detecta al probar la siguiente jugada; para comprobar que el rival
ganaría en su próximo turno hay que considerar también esa respuesta adversaria.
En clase, el LLM retador *desaprovecha* precisamente estas oportunidades:
reconocer patrones en un tablero no equivale a calcular las jugadas.


In [4]:
# Fábrica de una posición de prueba: crea un objeto nuevo cada vez para no compartir sus casillas.
def _immediate_win_for_max():
    b = Connect4()
    for col, p in [(0,MAX),(4,MIN),(1,MAX),(5,MIN),(2,MAX),(6,MIN)]:
        # b.push es un método; col y p son columna y jugador, obtenidos de cada tupla de la lista.
        b.push(col, p)
    return b            # la columna 3 completa cuatro en línea AHORA

def _immediate_threat_against_max():
    b = Connect4()
    for col, p in [(0,MAX),(3,MIN),(1,MAX),(3,MIN),(2,MAX),(3,MIN)]:
        # b.push es un método; col y p son columna y jugador, obtenidos de cada tupla de la lista.
        b.push(col, p)
    return b            # la columna 3 bloquea las tres fichas verticales de MIN

# assert lanza AssertionError si la política no elige la columna esperada para estas posiciones.
assert solution.choose_move(_immediate_win_for_max(), 500) == 3
print("ok  takes the immediate win (col 3)")
assert solution.choose_move(_immediate_threat_against_max(), 500) == 3
print("ok  blocks the immediate threat (col 3)")

ok  takes the immediate win (col 3)
ok  blocks the immediate threat (col 3)


## Garantía 1 — α-β devuelve el valor de minimax visitando menos nodos

La poda **no** es una aproximación: solo descarta ramas demostrablemente irrelevantes,
por lo que la *respuesta* es idéntica y solo cambia el *costo*. Reproducimos la tabla del
notebook con la posición que tiene dos fichas en la columna central.


In [5]:
# Devuelve una instancia nueva en una posición equivalente para comparar ambos algoritmos sin mezclar estado.
def fresh():
    b = Connect4(); b.push(3, MAX); b.push(3, MIN); return b

print(f"  {'depth':>6}{'minimax':>12}{'alpha-beta':>13}{'speedup':>10}")
print("  " + "-"*45)
for depth in (2, 4, 5, 6):
    # Las listas c1 y c2 contienen contadores mutables que comparten todas las llamadas recursivas.
    # Cuentan visitas, incluidas las hojas. Los tableros b1 y b2 son objetos independientes.
    b1, c1 = fresh(), [0]
    v1, _ = minimax(b1, depth, MAX, c1)
    b2, c2 = fresh(), [0]
    # La ventana empieza sin restricciones; CENTER_FIRST cambia el orden y favorece la poda.
    v2, _ = alphabeta(b2, depth, MAX, -inf, inf, c2, CENTER_FIRST)
    assert v1 == v2, f"values differ at depth {depth}"     # mismo VALOR
    assert c2[0] < c1[0]                                    # estrictamente menos nodos
    print(f"  {depth:>6}{c1[0]:>12,}{c2[0]:>13,}{c1[0]/c2[0]:>9.1f}x")
print("\nSame value every depth. Pruning changed the COST, not the ANSWER.")

   depth     minimax   alpha-beta   speedup
  ---------------------------------------------
       2          57           23      2.5x
       4       2,801          319      8.8x
       5      19,607        1,792     10.9x
       6     132,179        3,700     35.7x

Same value every depth. Pruning changed the COST, not the ANSWER.


## Garantía 2 — Observa su FALLO deliberado: el efecto horizonte

La profundidad *es* fuerza. Un agente de profundidad limitada juega de forma óptima
**con respecto a su horizonte**, pero pierde contra otro más profundo porque no ve una amenaza
que se resuelve una media jugada después de donde dejó de buscar.
El material destaca que aumentar la profundidad permite ver lo que quedaba fuera de ese horizonte.
Este es el momento de la semana en que observamos en vivo el fallo de una garantía.


In [6]:
# play crea el tablero y alterna turnos. Devuelve (ganador, objeto tablero final).
# _ descarta el tablero aquí; solo se comprueba qué jugador ganó cada partida.
w1, _ = play(2, 5)     # MAX a profundidad 2 frente a MIN a profundidad 5
w2, _ = play(6, 4)     # MAX a profundidad 6 frente a MIN a profundidad 4
print("play(2,5) winner:", "MIN (deep)" if w1 == MIN else w1)
print("play(6,4) winner:", "MAX (deep)" if w2 == MAX else w2)
assert w1 == MIN and w2 == MAX
print("\nGUARANTEE FAIL confirmed: the SHALLOW side loses every time — not by")
print("blundering, but by being blind past its horizon. Depth beats knowledge here.")

play(2,5) winner: MIN (deep)
play(6,4) winner: MAX (deep)

GUARANTEE FAIL confirmed: the SHALLOW side loses every time — not by
blundering, but by being blind past its horizon. Depth beats knowledge here.


## El LLM retador, sin interacción

En clase entra un LLM al torneo. Aquí usamos el backend determinista `echo`, sin interacción,
para que el notebook pueda ejecutarse en cualquier entorno. **`echo` es un SIMULADOR**:
no juega a Conecta 4 y su respuesta nunca debe reportarse como resultado de un modelo.
Ejecuta de nuevo con `backend="ollama"` o `backend="manual"` para usar un retador real.
El resultado esperado con un modelo real: jugadas legales *la mayor parte* del tiempo,
**victorias inmediatas desaprovechadas**, como no elegir la columna 3 cuando gana AHORA,
y un tiempo de respuesta unas 100 veces mayor que el de α-β.


In [7]:
from aicourse.llm import LLM
# POO: LLM es la clase y llm una instancia configurada con el backend echo.
# El objeto ofrece complete() para obtener una respuesta; echo no representa un modelo jugando.
llm = LLM(backend="echo")   # echo es un simulador sin interacción, NO una jugada real

# Las cadenas del prompt, los campos {board}/{legal} y las etiquetas se conservan para mantener el experimento.
PROMPT = '''Connect-4. You are 'X'. Columns numbered 0-6, left to right.
Board (bottom row last):
{board}
Legal columns: {legal}
Reply with one column number and nothing else.'''

b = _immediate_win_for_max()   # la columna 3 gana de inmediato
# str(b) invoca Connect4.__str__; b.legal_moves() consulta las columnas disponibles.
# PROMPT.format es un método de str; llm.complete es un método de la instancia LLM.
# El objeto devuelto guarda la respuesta textual en el atributo reply.text.
reply = llm.complete(PROMPT.format(board=str(b), legal=b.legal_moves()))
print("echo reply (FAKE):", reply.text)
print("\nα-β agent's move here:", solution.choose_move(b, 500), "(the winning col 3)")
print("Lesson: the searching agent proves the win; the LLM must be validated + retried,")
print("and in class it is the one that misses col 3. Re-run with a real backend to see it.")

echo reply (FAKE): [echo backend] 280 chars received. This is NOT a model response.

α-β agent's move here: 3 (the winning col 3)
Lesson: the searching agent proves the win; the LLM must be validated + retried,
and in class it is the one that misses col 3. Re-run with a real backend to see it.
